# FrimeRise Cinema Studio — Render na Nuvem (Google Colab)

Este notebook baixa automaticamente o seu repositório do GitHub, renderiza o vídeo na nuvem do Google (sem pesar no seu notebook) e salva o MP4 pronto no seu **Google Drive**.

### Como usar:
1. Ajuste as opções na **Célula 3** (na primeira vez, deixe `MODO = "demo"` para testar).
2. No menu lá em cima, clique em **Ambiente de execução → Executar tudo** (ou aperte **`Ctrl + F9`**).
3. Na primeira vez, clique em **Conectar ao Google Drive** quando aparecer a janela de permissão.
4. O vídeo pronto aparecerá aqui na tela e ficará salvo em `Meu Drive/FrimeRise_Exports/`.

## Passo 1 — Conectar seu Google Drive

In [ ]:
import os
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

PASTA_DRIVE = Path('/content/drive/MyDrive/FrimeRise_Exports')
PASTA_DRIVE.mkdir(parents=True, exist_ok=True)
print(f'Google Drive conectado! Seus vídeos serão salvos em: {PASTA_DRIVE}')

## Passo 2 — Baixar ou atualizar o seu projeto do GitHub

In [ ]:
import subprocess

REPO_URL = 'https://github.com/sml98/frimerise-cinema-studio.git'
REPO_DIR = Path('/content/frimerise-cinema-studio')

if REPO_DIR.exists():
    print('Atualizando repositório com git pull...')
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', 'origin', 'main'], check=True)
else:
    print('Clonando repositório pela primeira vez...')
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

# Garante que Pillow e NumPy estão instalados (FFmpeg já vem no Colab)
subprocess.run(['pip', 'install', '-q', 'pillow', 'numpy'], check=True)
print('Projeto pronto em:', REPO_DIR)

## Passo 3 — Escolher o vídeo para renderizar

- **`MODO = "demo"`**: gera o vídeo de demonstração com os 8 estilos documentais (ótimo para o primeiro teste).
- **`MODO = "episodio"`**: renderiza um JSON que você criou no Google AI Studio e subiu para o GitHub (ex.: `data/episodio.json`).

In [ ]:
MODO = 'demo'  # Troque para 'episodio' quando quiser renderizar seu JSON

# Usados apenas quando MODO = 'episodio':
ARQUIVO_JSON = 'data/episodio.json'
ARQUIVO_AUDIO = ''  # Opcional: caminho do áudio .wav/.mp3 dentro do repo ou Drive
NOME_SAIDA = 'station01_video.mp4'

print(f'Modo selecionado: {MODO} | Saída: {NOME_SAIDA}')

## Passo 4 — Renderizar, salvar no Google Drive e assistir aqui

In [ ]:
import shutil
from IPython.display import HTML
from base64 import b64encode

engine_script = REPO_DIR / 'engine' / 'doc_station01_engine.py'
if not engine_script.exists():
    raise FileNotFoundError(
        'Não encontrei engine/doc_station01_engine.py no repositório! '
        'Verifique se você já aplicou o frimerise_station01_pack.zip e deu git push.'
    )

saida_local = Path('/content') / NOME_SAIDA
cmd = ['python3', str(engine_script), '--no-vaapi', '--output', str(saida_local)]

if MODO == 'demo':
    cmd.append('--demo')
else:
    json_path = REPO_DIR / ARQUIVO_JSON
    if not json_path.exists():
        raise FileNotFoundError(f'Arquivo JSON não encontrado: {json_path}')
    cmd += ['--spec', str(json_path)]
    if ARQUIVO_AUDIO and os.path.exists(ARQUIVO_AUDIO):
        cmd += ['--audio', ARQUIVO_AUDIO]

print('Renderizando vídeo...')
subprocess.run(cmd, cwd=str(REPO_DIR), check=True)

destino_drive = PASTA_DRIVE / NOME_SAIDA
shutil.copy2(saida_local, destino_drive)
print(f'Pronto! Vídeo salvo no seu Google Drive em: {destino_drive}')

# Player embutido para assistir direto no Colab
mp4_bytes = saida_local.read_bytes()
data_url = 'data:video/mp4;base64,' + b64encode(mp4_bytes).decode()
HTML(f'<video width="720" controls><source src="{data_url}" type="video/mp4"></video>')